# 02 — Feature Extraction & Nail-Bed Analysis

## Objectives
1. Extract fingernail ROI from raw images
2. Analyze color-space distributions (RGB, LAB, HSV)
3. Compute per-channel statistics correlated with Hb levels
4. Segment nail bed region using adaptive thresholding

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

sns.set_theme(style="whitegrid")
%matplotlib inline

DATA_ROOT = Path("../data/raw")
METADATA_CSV = DATA_ROOT / "metadata.csv"

## 1. ROI Extraction

We detect the fingernail region using a combination of skin-color masking and contour detection.

In [ ]:
def extract_nail_roi(image: np.ndarray, margin: int = 10) -> np.ndarray:
    """
    Extract the fingernail region of interest from an image.
    
    Strategy:
    1. Convert to HSV and apply color thresholding for nail region
    2. Find largest contour (assumed to be nail bed)
    3. Crop with margin
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Nail bed typically has low saturation and moderate-to-high value
    lower = np.array([0, 20, 80])
    upper = np.array([25, 180, 255])
    mask = cv2.inRange(hsv, lower, upper)
    
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return image  # fallback: return full image
    
    # Largest contour = nail
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    
    # Add margin
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(image.shape[1], x + w + margin)
    y2 = min(image.shape[0], y + h + margin)
    
    return image[y1:y2, x1:x2]


print("ROI extraction function defined.")

## 2. Color-Space Analysis

Extract mean channel values in RGB, LAB, and HSV for correlation with Hb.

In [ ]:
def extract_color_features(roi: np.ndarray) -> dict:
    """
    Compute per-channel mean and std in RGB, LAB, and HSV color spaces.
    """
    features = {}
    
    # RGB
    for i, ch in enumerate(["R", "G", "B"]):
        features[f"rgb_{ch}_mean"] = roi[:, :, i].mean()
        features[f"rgb_{ch}_std"] = roi[:, :, i].std()
    
    # LAB
    lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB)
    for i, ch in enumerate(["L", "A", "B"]):
        features[f"lab_{ch}_mean"] = lab[:, :, i].mean()
        features[f"lab_{ch}_std"] = lab[:, :, i].std()
    
    # HSV
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    for i, ch in enumerate(["H", "S", "V"]):
        features[f"hsv_{ch}_mean"] = hsv[:, :, i].mean()
        features[f"hsv_{ch}_std"] = hsv[:, :, i].std()
    
    return features


print("Feature extraction function defined.")

## Next Steps

- Run feature extraction across the full dataset
- Compute Pearson/Spearman correlations between color features and Hb
- Feed top features into baseline regression models in `03_baseline_models.ipynb`